In [ ]:
# Install backend dependencies and pull the lightweight vision model
!sudo apt update && sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("Booting local Ollama server inside cloud space...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull qwen3.5:2b
!pip install ollama opencv-python

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:13 http://security.ubuntu.com/ubuntu

FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [ ]:
# 1. Update package lists and install zstd alongside the GPU tools
!sudo apt-get update
!sudo apt-get install -y zstd pciutils

# 2. Run the Ollama installer now that zstd is available
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Boot the Ollama background process cleanly using Python
import subprocess
import time

print("Booting local Ollama server inside cloud space...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Allow 5 seconds for the engine to initialize

# 4. Pull down the fast vision model and required packages
!ollama pull qwen3.5:2b
!pip install ollama opencv-python

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

##default webcam-code

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import ollama
import time
import io
from PIL import Image

# JavaScript function to inject into the browser to pull live frame data
def video_stream_js():
  js = Javascript('''
    var video;
    var canvas;
    var stream;

    async function initWebcam() {
      const div = document.createElement('div');
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;

      stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
    }

    async function captureFrame() {
      if (!video) await initWebcam();

      canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      return canvas.toDataURL('image/jpeg', 0.8);
    }

    window.captureFrame = captureFrame;
    ''')
  display(js)

# Inject the video streamer code into the runtime page
video_stream_js()

# --- CONFIGURATION ---
SAMPLE_INTERVAL = 10.0   # Grab a picture and analyze every 10 seconds
MODEL_NAME = 'qwen3.5:2b'
# ---------------------

print(f"Webcam stream listening... Running analysis loop every {SAMPLE_INTERVAL} seconds.")
print("Stop the execution of this cell manually to close the camera.")

try:
    while True:
        # Call JavaScript inside the browser to fetch the active webcam frame matrix data
        img_data = eval_js('captureFrame()')

        # Decode the base64 JPEG image back into a workable image file
        binary = b64decode(img_data.split(',')[1])
        temp_image_path = 'webcam_capture.jpg'
        with open(temp_image_path, 'wb') as f:
            f.write(binary)

        print(f"[{time.strftime('%H:%M:%S')}] Frame grabbed. Analyzing on T4 GPU...")

        start_time = time.time()
        # Query the local Ollama context model
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{
                'role': 'user',
                'content': 'Describe this image frame concisely in one clear sentence.',
                'images': [temp_image_path]
            }]
        )
        elapsed = time.time() - start_time

        # Print the descriptive text
        description = response['message']['content'].strip()
        print(f"Result ({elapsed:.2f}s): {description}")
        print("-" * 50)

        # Idle delay until next targeted sampling window
        time.sleep(SAMPLE_INTERVAL)

except KeyboardInterrupt:
    print("\nStream stopped by user.")

<IPython.core.display.Javascript object>

Webcam stream listening... Running analysis loop every 10.0 seconds.
Stop the execution of this cell manually to close the camera.
[05:02:27] Frame grabbed. Analyzing on T4 GPU...

Stream stopped by user.


##External webcam mode

In [ ]:
import subprocess
import time
import httpx

print("Checking and restarting Ollama Server...")

# Kill any ghost processes that might be blocking the port
!pkill ollama

# Relaunch the Ollama server process cleanly in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait and poll the port until the server actively responds
for i in range(10):
    try:
        httpx.get("http://localhost:11434")
        print("✓ Ollama Server successfully connected and listening!")
        break
    except httpx.ConnectError:
        print(f"Waiting for server to initialize... (Attempt {i+1}/10)")
        time.sleep(2)

Checking and restarting Ollama Server...
✓ Ollama Server successfully connected and listening!


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import ollama
import time

# JavaScript tracking script that targets non-default/external video sources
def video_stream_js():
  js = Javascript('''
    var video;
    var stream;

    async function initWebcam() {
      const div = document.createElement('div');
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;

      // Request a generic stream first to unlock device permissions/labels
      let temporaryStream = await navigator.mediaDevices.getUserMedia({video: true});

      // Find all connected video input hardware devices
      const devices = await navigator.mediaDevices.enumerateDevices();
      const videoDevices = devices.filter(device => device.kind === 'videoinput');

      // Close the temporary permissions stream
      temporaryStream.getTracks().forEach(track => track.stop());

      let targetDeviceId = null;

      // Strategy: Look for device strings containing "external", "usb", or anything that isn't the primary integrated unit
      if (videoDevices.length > 1) {
        console.log("Multiple cameras detected. Attempting to select external hardware device...");
        let externalCam = videoDevices.find(d =>
          d.label.toLowerCase().includes('usb') ||
          d.label.toLowerCase().includes('external') ||
          d.label.toLowerCase().includes('cam')
        );
        // Fallback: If labels are blank or don't match keywords, pick the second camera in the list
        targetDeviceId = externalCam ? externalCam.deviceId : videoDevices[1].deviceId;
      } else if (videoDevices.length === 1) {
        targetDeviceId = videoDevices[0].deviceId;
      }

      // Configure video constraints to bind strictly to our target external webcam identifier
      const constraints = {
        video: targetDeviceId ? { deviceId: { exact: targetDeviceId } } : true
      };

      stream = await navigator.mediaDevices.getUserMedia(constraints);
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
    }

    async function captureFrame() {
      if (!video) await initWebcam();

      canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      return canvas.toDataURL('image/jpeg', 0.8);
    }

    window.captureFrame = captureFrame;
    ''')
  display(js)

# Launch browser runtime hook
video_stream_js()

# --- CONFIGURATION ---
SAMPLE_INTERVAL = 10.0
MODEL_NAME = 'qwen3.5:2b'
# ---------------------

print(f"External webcam scanner listening... Sampling loop targets {SAMPLE_INTERVAL}s frequency.")

try:
    while True:
        img_data = eval_js('captureFrame()')

        binary = b64decode(img_data.split(',')[1])
        temp_image_path = 'webcam_capture.jpg'
        with open(temp_image_path, 'wb') as f:
            f.write(binary)

        print(f"[{time.strftime('%H:%M:%S')}] Frame intercepted from external camera. Sending to T4 GPU...")

        start_time = time.time()
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{
                'role': 'user',
                'content': 'Describe this image frame concisely in one short sentence.',
                'images': [temp_image_path]
            }]
        )
        elapsed = time.time() - start_time

        description = response['message']['content'].strip()
        print(f"Result ({elapsed:.2f}s): {description}")
        print("-" * 50)

        time.sleep(SAMPLE_INTERVAL)

except KeyboardInterrupt:
    print("\nStream halted.")

<IPython.core.display.Javascript object>

External webcam scanner listening... Sampling loop targets 10.0s frequency.
[05:13:49] Frame intercepted from external camera. Sending to T4 GPU...
Result (10.13s): A man with glasses and a beard sits in a dimly lit room wearing a white striped shirt.
--------------------------------------------------
[05:14:09] Frame intercepted from external camera. Sending to T4 GPU...
Result (5.85s): A man wearing glasses and a striped shirt holds a small pink figurine up to the camera in his room.
--------------------------------------------------
[05:14:26] Frame intercepted from external camera. Sending to T4 GPU...
Result (6.23s): A young man with curly hair, a beard, and glasses sits indoors wearing a white striped shirt, looking directly at the camera.
--------------------------------------------------
[05:14:42] Frame intercepted from external camera. Sending to T4 GPU...
Result (7.69s): A young man with curly hair and glasses rests his chin on his hand while looking at the camera, wearing a